In [54]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from src import inputs, firms_api, cleaning, clustering, map_wf, wfss

API_KEY = firms_api.get_api_key()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
API-Key: 3d1ef73c3c932e5f3738d87f205a9cec


# Visualise Wildfires (FIRMS API)
by Paul Ghisletti, May 18th 2026
***

## What is FIRMS?

**NASA FIRMS (Fire Information for Resource Management System)** provides near real-time satellite observations of active fires and thermal anomalies worldwide. The system aggregates wildfire detections from sensors such as MODIS and VIIRS and makes them accessible through APIs, maps, and downloadable datasets.

Typical FIRMS datasets contain:

- geographic coordinates (latitude and longitude)
- acquisition date and time
- fire radiative power (FRP)
- confidence scores
- satellite and instrument identifiers
- brightness temperature measurements
- detection type classifications
- optional metadata such as day/night flags and scan geometry

Each row usually represents a single satellite fire detection pixel rather than an individual wildfire event. Multiple nearby detections may therefore correspond to the same wildfire.
***
## Instructions
1. Set up your API-key with the instructions below.
2. Hit `Run All` to start running the cells in this notebook
3. Define the parameters of the API query by typing your desired inputs in the widgets below. If you have already defined some parameters, entering 'x' as input keeps the old parameter. The parameter are:
    - area: what area would you like the data to cover?
    - date: from when should the data be?
    - sensor: what sensor should provide the data?
    - number of days: what timespan should your data cover?
    - open in browser: you can decide whether the map should automaticall be opened in your browser.
    - Shoould an error be raised during parameter input, apply its suggestions to your input and run the notebook from that cell downwards.
4. Hit enter after each input to continue the process.
5. Once all parameters are set and you have read through the information about the map, hit enter to generate the map automatically.
***
## API-key Setup
Make sure you followed the steps 2.1. to 2.3. in the `README.md` file on how to set your individual API-key: Check, whether the following output matches your individual API-key from FIRMS. Here, you can also see how many free transactions you have left with your api-key

***
## Input Parameters
### 1. Area
There are 4 possible input types for this parameter:
- Country:
    - type a country's name e.g. 'Algeria'. Common alternatives should work too, e.g. 'Burma' for Myanmar.
    - ISO 3-letter country code, e.g. 'AUS' for Australia or 'GER' for Germany
- Continent: choose from one of the 7 continents.
- 'World': for global coverage, use this. This is also the default value if you do not input anything.
- Bounding Box: if you have a more specific area in mind, pass a list as input with the format: `[west, south, east, north]`

In [55]:
inputs.ask_area()

area set to: world


### 2. Date
Type your date in the format: **YYYY-MM-DD**. If you wish the **most recent data, leave this cell empty**.

In [56]:
inputs.ask_date()

date kept as: 2020-01-01


### 3. Sensor
There are 3 Satellite instruments offered by FIRMS, split into 4 sensors. For most sensors, there are 2 datasets, one for near real time data (usually current day) and standard processing (preprocessed by FIRMS, but containing more information. Latency usually months)
| Dataset          | Resolution | Coverage | revisit time        | Latency              | Best For                     |
| ---------------- | ---------- | -------- | ------------------- | -------------------- | ---------------------------- |
| LANDSAT_NRT      | 30 m       | US/CAN   | ~ 16 days           | Near real-time       | Detailed fire mapping        |
| MODIS_NRT        | 1 km       | Global   | ~ 0.25 days         | Near real-time       | Large active fires           |
| MODIS_SP         | 1 km       | Global   | ~ 0.25 days         | Standard processing  | Historical analysis          |
| VIIRS_NOAA20_NRT | 375 m      | Global   | ~ 0.5 days          | Near real-time       | Small/medium fires           |
| VIIRS_NOAA20_SP  | 375 m      | Global   | ~ 0.5 days          | Standard processing  | Higher-quality archive       |
| VIIRS_NOAA21_NRT | 375 m      | Global   | ~ 0.5 days          | Near real-time       | Newest VIIRS stream          |
| VIIRS_SNPP_NRT   | 375 m      | Global   | ~ 0.5 days          | Near real-time       | General wildfire monitoring  |
| VIIRS_SNPP_SP    | 375 m      | Global   | ~ 0.5 days          | Standard processing  | Historical wildfire analysis |  


Also note, that different sensors and datasets provide different time spans:

In [57]:
availability_all_df = firms_api.get_availability_all(API_KEY)
display(availability_all_df.head(8))

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-22
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-22
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-22
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-22
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-22


For the sensor input, just copy paste one of the options from the table above:

In [58]:
inputs.ask_sensor()

sensor kept as: VIIRS_NOAA20_SP


### 4. Number of Days
The FIRMS API allows up to 5 days. The range begins with the date set above and counts n days to the future. However, nevermind if you put not date expecting the most recent data and now enter 5, since this contradiction is handled inside the functions.

In [59]:
inputs.ask_n_days()

n_days kept as: 5


### 5. Open in Browser
You can choose to open the .html file (which stores the visualisation data) in your browser for a larger panel:  
- `True`: Open in Browser (larger interface)
- `False`: Open in Notebook (smaller interface)
- If left empty, it will open in Notebook

In [60]:
inputs.ask_browser()

browser kept as: True


### 6. Save Map
You can toggle saving the map to the current working directory. Either enter 'True' if you want to save it as an .html file or 'False' if you don't. If you chose to open the map in the browser, saving it is required. However, this dependency is handles, regardless of what you enter below.

In [61]:
inputs.ask_save()

save kept as: True


### 7. Max Rows
Because datasets that cover the entire world for 5 days can contain upwards of 100'000 entries, it would be cimputationally too demanding to render all those points several times (once for each point layer). To prevent the program from crashing or long computing times, you set the maximum number of entries that are rendered in the final map. A random subsample of the original dataset is taken only for rendering the map.
- 5'000: easy computing, quick visualisation. But: on a large scale, a lot of detail is lost, to a point where smaller fires might disappear completely.
- 10'000: middle ground
- 20'000: very long computation, might crash the program if too many maps are generated.  

If left empty, it will default to 5'000.

In [62]:
MAX_ROWS = inputs.ask_max_rows()

max_rows kept as: 5000


## Visualisation
In general, if you need more information on map layers, scores etc., consult the wwildfires.ipynb, where the process of creating this project is documented in detail.
### Map Layers:
- Area outlines
- Heatmap (weighted by FRP)
- Fires (clustered pixels)
- Severity Score
- Fire pixels by Fire Radiative Power in megawatts
- Fire pixels by detection time
- Volcanoes
- Offshore fires
- Other landbased fire sources  

Depending on the sensor/dataset you chose, only select layers are available and displayed:  
|                   |Area outlines  |Heatmap    |Fires (clustered)  |Severity score |Fire pixels by FRP |Fire pixels by datetime    |Volcanoes  |Offshore       |Other land source
|-------------------|---------------|-----------|-------------------|---------------|-------------------|---------------------------|-----------|---------------|----------
|LANDSAT_NRT        |       x       |     x     |         x         |               |                   |              x            |           |               |           
|MODIS_NRT          |       x       |     x     |         x         |               |         x         |              x            |           |               |           
|MODIS_SP           |       x       |     x     |         x         |               |         x         |              x            |     x     |       x       |     x     
|VIIRS_NOAA20_NRT   |       x       |     x     |         x         |               |         x         |              x            |           |               |           
|VIIRS_NOAA20_SP    |       x       |     x     |         x         |       x       |         x         |              x            |     x     |       x       |     x     
|VIIRS_NOAA21_NRT   |       x       |     x     |         x         |               |         x         |              x            |           |               |           
|VIIRS_SNPP_NRT     |       x       |     x     |         x         |               |         x         |              x            |           |               |           
|VIIRS_SNPP_SP      |       x       |     x     |         x         |       x       |         x         |              x            |     x     |       x       |     x    

### Clusters
FIRMS delivers wildfire data in the form of pixels that detect fire. So one entry in the dataset does not equal to one wildfire. Therefore, we cluster these pixels, that detected fire by their location. With these clusters, we try to represent individual fires or fire fronts. However, this is not perfect, since we are using a hard coded distance to basically distinguish one cluster from another, it might happen that one cluster represents several fires, which are just close together, or one fire is split into 2 clusters because some pixels could be missing.
### FRP (Fire Radiative Power)
Fire Radiative Power (FRP) is widely used in wildfire remote sensing as an estimate of fire intensity. It measures the rate at which a fire releases thermal energy, allowing researchers to compare wildfire activity across regions and time periods.
### Severity Score
Note that this feature is only available for the `VIIRS_NOAA20_SP` and `VIIRS_SNPP_SP`datasets.  

The wildfire severity score (WFSS) is an attempt at quantifying the severity of a wildfire. It is calculated based on the fire's extent (larger fires tend to score higher), FRP of the entire fire (higher FRP sum tends to score higher) and FRP mean (again, a higher mean tends to score higher). The WFSS uses a logarithmic scale starting at 1. As a reference, the Canadian Wildfires 2023 score around 10, the Australian Bush Fires 2019/2020 score around 16. The scale is capped at 20 (due to its logarithmic nature).
### Constraints of the map
 - the time plotting is only useful for n_days more than just 1 or 2, since the color ramp is normalised by max time span and not absolute time difference, so if there is only a couple of hours between different observations the coloring might just pick up satellite movement instead of fire movement.
 - For timespans of more than 5 days, you need to rerun the program several times. Each time rendering a new map, which is not ideal for comparison.
### Handling missing API data
In the case of the Australian Bush Fires 2019/2020, for example, the API does not allow you to gather data from only that region. This can be circumvented by downloading data for the entire world instead. But this results in many more points being excluded due to the random sub sampling. Therefore we can clip the data manually after calling the API adn receiving the WildFireQuery object. Follow these steps if that is the case:
1. In the Section called **!!! Fail-Safe !!!** below, change `execute`to `True`
2. Enter your desired area (e.g. Australia)
3. Rerun the entire notebook and use 'world' for area instead of your desired area.
### Generate the Map
If you are ready, press `Enter`on your keyboard to generate the map.
The following code should run automatically and open the map. The output is mainly thought for debugging.

In [86]:
inputs.generate_map()

In [87]:
PARAMS = inputs.generate_query_parameters()
inputs.check_params(PARAMS)
WF = firms_api.area_api_query(firms_api.get_api_key(), **PARAMS)

API-Key: 3d1ef73c3c932e5f3738d87f205a9cec
Sensor input: VIIRS_NOAA20_SP
Area input: world
Area bbox: world
N_days input: 5
Date input: 2020-01-01
All input correct
URL: https://firms.modaps.eosdis.nasa.gov/api/area/csv/3d1ef73c3c932e5f3738d87f205a9cec/VIIRS_NOAA20_SP/world/5/2020-01-01
Shape (rows, columns): (308939, 16)
Our current transaction count is 380/5000


In [88]:
firms_api.test_wf_empty(WF)

### **!!! Fail-Safe !!!**

In [89]:
execute = True
area = 'australia'

if execute:
    WF.data = firms_api.clip(WF.data, area)
    WF.geometry = firms_api.get_geometry(area)

In [90]:
WF.cleaned = cleaning.clean(WF.data)

=== Cleaning Summary ===
Missing required fields : 0
Invalid coordinates     : 0
Invalid FRP             : 182
Invalid confidence      : 0
Low confidence          : 9072
Invalid daynight        : 0
Invalid type            : 0
Duplicates              : 0
-------------------------
Rows removed: 9254 of 101439
Rows remaining: 92185


In [91]:
firms_api.test_max_rows(WF)

/Users/paulghisletti/Desktop/SDS210/wildfires/SDS210-wildifres-project/src/firms_api.py:357: UserWarning: Dataset too large for individual point plotting (101439 rows). A random subset with n=5000 is displayed instead
  warnings.warn(


In [92]:
WF.clustered = clustering.cluster_wf(WF)

/Users/paulghisletti/Desktop/SDS210/wildfires/SDS210-wildifres-project/src/clustering.py:137: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = clustered.dissolve(by='cluster').centroid.rename('geometry')


In [93]:
weights = [0.4, 0.1, 0.5] # [frp sum, frp mean, cluster size]
WF.clustered = wfss.severity_score(WF, weights)

In [95]:
import webbrowser
import os
MAP_PARAMS = inputs.generate_mapping_parameters(WF)
m = map_wf.map_wf(WF, MAX_ROWS, True)
output_path = output_path = os.path.join(os.getcwd(), f"wildfire_{WF.area_display}_{WF.date}.html")
if MAP_PARAMS["save"] == True:
        m.save(output_path)
if MAP_PARAMS["browser"] == False:
    display(m)
if MAP_PARAMS["browser"] == True:
    webbrowser.open(f"file://{output_path}")